# DATASCI 350 - Data Science Computing

## Assignment 09 - Parallel computing and SQL

### Instructions

This assignment evaluates your understanding of lectures 21 and 22: serial and parallel execution, `joblib`, Amdahl's law, Dask, storage formats, and SQL with DuckDB.

Some tasks are reading and reasoning about material you already have. Most ask you to run code and explain what came back.

Eight tasks ask you to run code: tasks 03, 04, 06, 07, 08, 09, 10 and 11. Install what they need with `pip install joblib numpy duckdb pandas dask`. Tasks 03 and 04 use the `%timeit` magic, so run them inside Jupyter or IPython rather than in a plain `python` script.

Tasks 06 to 11 use one file, `wdi_panel.parquet`, which is 450 KB. Download it once with the `curl` line in task 06 and reuse it for the other five. Nothing else in this assignment downloads anything.

Your timings will not match mine, and they do not need to. Core counts, memory and background processes differ between machines. Report what your machine did, paste the actual output, and answer the questions about the numbers you got. Do not edit outputs.

You must complete this assignment individually. You may use notes, books, and AI tools, but you must submit original work. I strongly recommend that you do not use AI tools to generate your solutions, as this will not help you learn the material. Acknowledge all resources used, including input from classmates and AI, in a short note at the end of your submission. If you are unsure about permissible resources or proper acknowledgement, please ask the instructor.

### Submission

Please submit your solutions as either a single Jupyter notebook or a PDF file. For each task, paste the commands you ran, any code you wrote, the output you got, and your written answer where the task asks for one. Screenshots are welcome but not required if you paste the text. Submit your completed assignment to Canvas.

### Task 01

Here are three jobs.

1. Resizing 400 holiday photographs to thumbnails. Each photograph is a separate file and none of them depends on another.
2. Computing a running total over a column of 400 numbers, where each entry is the previous total plus the next number.
3. Fetching 200 web pages and saving the HTML of each one.

For each job, say which fits best: embarrassingly parallel, the kind of job threads help with, or neither. A job can fit more than one; if so, say which and why. One line per job is enough.

Then answer in a few sentences: pick the job that does not benefit from either approach and say what about it blocks parallel work.

Hint: the "Serial vs parallel execution" slide of lecture 21 defines embarrassingly parallel, and the `concurrent.futures` slide separates CPU-bound from I/O-bound work.

*Commands, output, and answer here.*

### Task 02

Amdahl's law says the speed-up you can get from `N` cores is

```text
speedup = 1 / (s + p / N)
```

where `s` is the share of the program that must run serially and `p = 1 - s` is the share that can be split across cores.

You have profiled a program and found that a quarter of its runtime is serial: `s = 0.25`, so `p = 0.75`.

Compute the speed-up for each of these, showing the arithmetic, not just the final number, and round each speed-up to two decimal places:

- 4 cores
- 8 cores
- 32 cores
- infinitely many cores

Then answer in a few sentences: which of the two jumps buys you the least extra speed per added core, 4 to 8 cores or 8 to 32 cores? Show the division. Given the ceiling you computed for infinitely many cores, say what you would work on instead of buying more cores.

Hint: the "Amdahl's law: the speed-up ceiling" slide of lecture 21 works through the same arithmetic with `s = 0.1` on 8 cores.

*Commands, output, and answer here.*

### Task 03

This task times the same work three ways on your own machine. Run it in Jupyter or IPython, because `%timeit` is a notebook magic and does not work in a plain script.

Start by reporting your core count:

```python
import os
print(os.cpu_count())
```

Then define the function from the "Serial vs parallel execution" slide of lecture 21:

```python
import numpy as np
from joblib import Parallel, delayed

def calculation(size=10000000):
    # Create a large array and perform operations
    arr = np.random.rand(size)
    for _ in range(10):
        arr = np.sqrt(arr) + np.sin(arr)
    return np.mean(arr)
```

Now time three things, one cell each, and paste all three results:

```python
%timeit calculation()
```

```python
%timeit [calculation() for _ in range(4)]
```

```python
%timeit Parallel(n_jobs=4)(delayed(calculation)() for _ in range(4))
```

Then answer two things.

1. Divide your serial time for four calls by your parallel time for four calls, and report that number as your speed-up. Show the division.
2. In a few sentences, say why the speed-up is less than 4 even though you asked for four workers, and name two things joblib has to do that the serial version does not.

Your numbers will differ from mine and from your classmates'. That is expected, and you are graded on the reasoning, not the milliseconds.

Hint: the "Serial vs parallel execution" slide of lecture 21 runs exactly these three timings and the last bullet on the right names the two costs.

*Commands, output, and answer here.*

### Task 04

This task turns a serial pipeline into a Dask pipeline. Run it in Jupyter or IPython.

Here are three plain functions and a loop that runs them for four random seeds. Each seed simulates two million numbers, standardises them, and counts how many land more than three standard deviations from the mean.

```python
import numpy as np
import dask

def simulate(seed):
    rng = np.random.default_rng(seed)
    return rng.normal(loc=50, scale=10, size=2_000_000)

def standardise(x):
    return (x - x.mean()) / x.std()

def count_extremes(z):
    return int((np.abs(z) > 3).sum())

seeds = [1, 2, 3, 4]
```

First, run the serial version once in a normal cell, then print `counts` in the next cell:

```python
counts = []
for seed in seeds:
    x = simulate(seed)
    z = standardise(x)
    counts.append(count_extremes(z))
```

Then time it: copy the same loop into a new cell and put `%%timeit` on its first line (two `%` signs, because it times a whole cell). Variables created inside a `%%timeit` cell do not survive it, which is why you print `counts` from the untimed run. Paste the printed counts and the timing.

Now write the Dask version yourself. Make each of the three functions a delayed task, build the same pipeline for the same four seeds without running anything, then run all of it with one call and time that call with `%timeit`. Paste your code, the result of the call, and the timing.

Then answer three things.

1. Are the four counts the same in both versions? Say why they have to be.
2. The pipeline has twelve tasks. How many can run at the same moment, and what stops the others from starting?
3. Suppose you added a fourth function that takes all four counts and returns their sum. Say what that does to the shape of the graph, and whether it changes your answer to question 2.

Hint: the two "Dask delayed" slides of lecture 21. The first shows the decorator, the second a three-stage pipeline like this one.

*Commands, output, and answer here.*

### Task 05

Lecture 21 wrote one month of generated data twice, once as 30 CSV files and once as a single Parquet file, then ran the same group-by on each. The numbers were:

- Size on disk: 182 MB as CSV, 86 MB as Parquet
- Group-by time: 741 ms on the CSV files, 66.8 ms on the Parquet file

Answer in a few sentences, treating the two gaps separately.

1. The size gap: name two properties of Parquet that make the same data smaller on disk.
2. The speed gap: the group-by touched two columns out of the several in the file. Explain why that matters for Parquet and not for CSV. Say whether the file being smaller is enough on its own to explain a speed-up of about eleven times.

Then name one situation where you would still hand somebody a CSV, and say why Parquet would be the wrong choice there.

Hint: the "Reading and writing data" and "Why Parquet?" slides of lecture 21.

*Commands, output, and answer here.*

### Task 06

This task and the five after it use the WDI panel from lecture 19: 59,024 rows of country, ISO3 code, indicator, year and value. Download it once, into whatever folder you are working in:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-19/data/wdi_panel.parquet
```

Keep it next to your notebook, then create the view the later tasks use:

```python
import duckdb

duckdb.sql("""
    CREATE OR REPLACE VIEW wdi AS
    SELECT * FROM 'wdi_panel.parquet'
""")
```

Now write your own queries.

1. For the year 2020, return one row per indicator with three columns: the indicator, the number of rows, and the number of rows that have a value. Sort by the number of values, lowest first. Paste the query and the result.
2. One indicator has no values at all in 2020. Compute the mean of `value` for that indicator in 2020, and paste the query and the result. Say what you got and why SQL gave you that rather than an error.
3. Find the last year in which that indicator has at least one value, and how many countries have a value in that year. Paste the queries and the results.

Then answer in one or two sentences: what would you check before reporting any mean from this panel?

Hint: the "COUNT(\*) and COUNT(value)" and "Missing values" slides of lecture 22.

*Commands, output, and answer here.*

### Task 07

Keep the `wdi` view. A classmate wants to count the missing values in the whole panel and writes this:

```python
duckdb.sql("SELECT COUNT(*) FROM wdi WHERE value = NULL")
```

Run it and paste the result. Then fix the query so that it counts the rows where `value` is missing, run it, and paste the result.

Then answer in a few sentences: why does the first query return 0 when the panel clearly has missing values? Say what `value = NULL` gives for a row that has a value and for a row that does not.

Hint: the "Missing values" slide of lecture 22.

*Commands, output, and answer here.*

### Task 08

Keep the `wdi` view. This question is about how many rich countries there were in each recent year.

Write one query that, for the `gdp_per_capita` indicator:

1. Counts how many countries had a value above 30,000 in each year from 2010 to 2023.
2. Keeps only the years where that count is above 40.
3. Returns the year and the count, ordered by year.

Paste the query and the result table. You should get fewer than fourteen rows: at least one year does not clear the threshold.

Then answer in a few sentences. Two of the conditions you wrote are filters: "value above 30,000" and "count above 40". Say which clause you put each one in, and explain why they cannot be swapped.

Hint: the "WHERE and HAVING" slide of lecture 22 uses `WHERE` on a row condition and `HAVING` on an aggregate in the same query. The "COUNT(\*) and COUNT(value)" slide shows counting inside a `GROUP BY`.

*Commands, output, and answer here.*

### Task 09

Keep the `wdi` view. The "CASE" slide of lecture 22 put seven countries into income bands. This task does the same for every country and counts them.

Write one query that, for the `gdp_per_capita` indicator in 2020, puts each country into one of these five groups and returns one row per group with the number of countries in it, largest group first:

- `No data`: the value is missing
- `High income`: 13,846 and above
- `Upper middle`: 4,466 and above
- `Lower middle`: 1,136 and above
- `Low income`: everything else

Paste the query and the result. The five counts should add up to 217.

Then answer two things.

1. Delete the `No data` line from your `CASE` and run the query again. Paste the result, and say which group grew, by how many, and why.
2. In one sentence: why is the result of question 1 wrong even though the query ran without an error?

Hint: the "CASE" and "GROUP BY" slides of lecture 22. You can group by the name you give the `CASE` column.

*Commands, output, and answer here.*

### Task 10

Keep the `wdi` view. This task ranks countries by life expectancy in 2000 and in 2020, then looks at five of them.

Write a query that returns four columns for Brazil, China, India, Japan and the United States: the year, the country, the life expectancy rounded to one decimal place, and the country's rank in that year among all 217 countries, highest first. Return the years 2000 and 2020 only, sorted by country and then year. You should get ten rows.

The ranks must be computed among all 217 countries, not among the five. That means ranking first and choosing the five countries afterwards. The "DESCRIBE and SUMMARIZE" slide shows the pattern: a query can select from the result of another query written inside brackets.

Paste the query and the result.

Then answer three things.

1. Move the five-country condition into the `WHERE` clause of the ranking query itself and run it again. Paste the result and say what happened to the ranks, and why.
2. The United States' life expectancy went up between 2000 and 2020, and its rank got worse. Say how both can be true.
3. Pick one other country from your result and say in one sentence what its two ranks tell you.

Hint: the "Window functions" and "Ranking with a window" slides of lecture 22.

*Commands, output, and answer here.*

### Task 11

This task joins two small tables. Keep the `wdi` view from task 06, then run these four statements from the "Two small tables" slide of lecture 22.

```python
duckdb.sql("""
    CREATE OR REPLACE TABLE gdp2020 AS
    SELECT country, iso3, ROUND(value, 0) AS gdp_per_capita
    FROM wdi
    WHERE indicator = 'gdp_per_capita' AND year = 2020
      AND iso3 IN ('USA','BRA','CHN','IND','DEU','JPN')
""")
```

```python
duckdb.sql("""
    CREATE OR REPLACE TABLE regions
        (iso3 VARCHAR PRIMARY KEY, region VARCHAR)
""")
```

```python
duckdb.sql("""
    INSERT INTO regions VALUES
        ('USA', 'North America'), ('BRA', 'Latin America'),
        ('CHN', 'East Asia'),     ('IND', 'South Asia'),
        ('DEU', 'Europe'),        ('KOR', 'East Asia'),
        ('AUS', 'Oceania')
""")
```

```python
duckdb.sql("SELECT * FROM regions ORDER BY iso3")
```

There are six rows in `gdp2020` and seven in `regions`. `JPN` has a GDP figure and no region. `KOR` and `AUS` have a region and no GDP figure.

Write one query that lists every country with a 2020 GDP figure, with three columns: the country, its GDP per capita, and its region. Every country in `gdp2020` must appear, Japan included. Where the region is missing, show `No region` instead of an empty cell. Sort by GDP per capita, highest first.

Paste the query and the result. You should get six rows.

Then answer in a few sentences. Which table has to be the one named in `FROM` this time, and why? Korea and Australia are in `regions` but not in your result: why not? What did `COALESCE` change?

Hint: the "LEFT JOIN" and "Missing values" slides of lecture 22. Appendix 01 answers a different question: there, every region had to survive.

*Commands, output, and answer here.*